**Fin 585**  
**Diether**  
**Winter 2026**  
**Diether Exam: Empirical**  
<br>

**Overview**

For the empirical part of the exam, you revisit the short-selling data you used in one of you early programming assignments. First, you need to create lagged loan fee based portfolios and test the four factor model using these portfolios. Second, you will create double sort portfolios formed on lagged loan fee and on past analyst dispersion (where analyst dispersion is measured as in Diether, Malloy, and  Scherbina (2002)). In this exam, you will form the analyst dispersion bins based on average dispersion from the past two months.

You are allowed to use the Fin 585 library on this exam. **This portion of the exam is open course materials.** Of course, everything must be your own work; you cannot collaborate or work with anyone.

The exam is due on Thursday, March 5 at 2:00 PM. The exam should be turned in via Learning Suite.
<br><br> 

**Review: Short Selling Background**

When someone shorts a stock, they profit if the stock goes down instead of going up, but short-selling transactions are more complicated than going long (buying a stock and then later selling it).  There are four basic steps to short selling:

1. *The short seller borrows the desired number of shares from someone.* This is usually done by the broker who locates the shares and the broker becomes the middleman for the short seller and the lender (note, the broker is often both the middleman and the lender). The lender expects to be paid interest on the loan which is the main cost of shorting. The loan is callable by the lender at any time. The short seller can repay the loan at any time. 

2. *The short seller sells the shares.* The proceeds are put into an interest-bearing account called the collateral account. Most lenders require the collateral account to contain 102% of the value of the proceeds.  The collateral account usually invests in low risk, short term securities. When the short seller borrows the stock there are lending fees; the short seller pays interest on the loan. Typically, the interest rate is small. The overall interest rate earned on the collateral account is split between the lender and the short seller. The portion of the interest rate received by the short seller is called the rebate rate. The **loan fee** is the portion paid the lender, and is equivalent to the interest rate the short seller pays on the loan. Therefore, the **loan fee** is the main direct cost of shorting. There can be a zero or negative rebate rate; a negative rebate rate corresponds to a situation where the lender receives all the interest in the collateral account and the short sellers pays additional interest out of her pocket to the lender.

3. *Pay any dividends while the loan is open.* The short seller must pay to the lender the cash equivalent of any dividends paid out on the stock.

4. *Buy the shares back.* The short sellers profits are the following:

$$
    Profit = Sell - Buy -(Interest \ Paid)
$$
 
**Data**

The are two main datasets for this exam. The first is the monthly CRSP data augmented with short-selling fees. Note, it's not exactly the same as the data in the first homework (e.g., no variables have been lagged for you). The data are monthly stock data for all stocks in the U.S. from May of 2002 to August of 2012. The basic unit of observation is the stock-month. You can download the data directly using the following link: [the data](https://diether.org/prephd/14-mstk_short.csv). There is also a link on *Learning Suite*. The data contain the following variables:

|Variable | Description                                       |
|---------|---------------------------------------------------|
|permno   | stock identifier                                  |
|caldt    | calendar date                                     |
|cusip    | another stock identifier                           
|mdt      | calendar date unique at the year-month level      |
|ret      | monthly return                                    |
|prc      | stock price                                       |   
|me       | market equity                                     |
|fee      | the loan fee expressed a percent per anum         |
|shrcd    | CRSP share code                                   |


Note, if the loan fee is negative, it is a data error. 

The second primary data source is from I/B/E/S. It contains monthly observations for variables related to analyst earnings per share forecasts.

|Variable | Description                                              |
|---------|----------------------------------------------------------|
|cusip    | stock identifier for the I/B/E/S data                    |
|mdt      | calendar date unique at the year-month level             |
|disp     | month end analyst dispersion as defined in DMS (2002)    |
|numest   | number of analyst forecasts used to compute disp         |

Just like in the CRSP data, `mdt` reports the date as if its from the first day of a month, but the observations are actually observations from the end of the trading month. So `disp` for `cusip = '39040610'` and `caldt = '1990-04-01'` is analyst dispersion (as computed in DMS (2002) as of the last trading day in April 1990. You can download the data from the following link: [Monthly I/B/E/S data](https://diether.org/prephd/14-ibes.csv).<br><br>


**Tasks and Questions**  

1. Construct three equal-weight portfolios using lagged fee as a the criterion variable. Portfolio 0: an equal-weight portfolio the includes all stocks with lagged fee less than or equal to 1% (loan fees are expressed as the interest rate per anum). Portfolio 1: an equal-weight portfolio the includes all stocks with lagged fee greater than 1% and less than or equal to 4%. Portfolio 2: an equal-weight portfolio the includes stocks with lagged fee greater than 4%. Make your portfolio sample period be January 2003 to July 2012. Also, include a spread portfolio in your summary statistics (including a t-test of whether the average return is statistically different from zero for each portfolio).

2. Test whether the four factor model holds with respect to the lagged loan fee portfolios formed in question #1. You should summarize your regressions results in one table. Explain what you can infer about validity of the four factor model based on these regression results. You can download the factors from the following link: [Factor Data](https://diether.org/prephd/14-factors.csv).

3. What do the factor loadings from the four factor model regression about portfolio 2 (high lagged loan fee portfolio) tell you about the characteristics (small-cap/large cap,value/growth, etc) of the stocks in the high loan fee portfolio. Explain.

4. Form portfolios double sorted on lagged loan fee and average analyst dispersion from the previous two months. You should use independent sorts to create these double sort portfolios (as in Fama and French (1992)). The analyst dispersion breakpoints should be *terciles* (A set of data arranged in order with values that partition the data into three groups, each containing one-third of the total data.). For the loan fees use the three portfolio categories you created in question #1. Therefore, your program should create **9 portfolios**. Report summary statistics for your portfolios (including a t-test of whether the average return is statistically different from zero for each portfolio). Make your portfolio sample period be January 2003 to July 2012. Note, some of your portfolios may contain a few missing values. Don't worry about them. The regression estimation in the next questions should work just fine.

5. Test whether the *four* factor model holds with respect to the double sort portfolios in question #3. You should summarize your regressions results in one table. Explain what you can infer about validity of the four factor model based on these regression results.

6. Do you think Questions #4 and #5 represent a good test of the Miller (1997) model? Explain why or why not? If yes, are the results in questions #4 and #5 consistent with the Miller model?

7. DMS's (2003) best test of the Miller model is when they create portfolios double sorted on size (lagged market-cap) and analyst dispersion. Give one reason why double sort portfolios formed on loan fees and analyst dispersion may represent a superior test than double sort portfolios formed on size and analyst dispersion, and one reason it may represent an inferior test.<br><br>

In [198]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

In [199]:
df = pd.read_csv("14-mstk_short.csv",parse_dates=['caldt','mdt'])
ibes = pd.read_csv("14-ibes.csv",parse_dates=['mdt'])

In [200]:
df["prclag"] = df.groupby('permno').prc.shift()
df["melag"] = df.groupby('permno').me.shift()
df["feelag"] = df.groupby('permno').fee.shift()

df = df[('2003-01-01' <= df.caldt) & (df.caldt < '2012-08-01') & (df.feelag >= 0)]

df.reset_index(drop=True)

,permno,caldt,cusip,mdt,ret,prc,me,fee,shrcd,prclag,melag,feelag
0,10001,2005-06-30,29274A10,2005-06-01,0.128430,9.05,26.363,0.32701,11,8.02,21.053,0.15000
1,10001,2005-07-29,29274A10,2005-07-01,0.009945,9.14,26.625,0.15000,11,9.05,26.363,0.32701
2,10001,2005-08-31,29274A10,2005-08-01,0.039387,9.50,27.674,NaN,11,9.14,26.625,0.15000
3,10001,2005-10-31,29274A10,2005-10-01,-0.119040,10.10,29.421,0.15000,11,11.51,33.529,0.16250
4,10001,2005-11-30,29274A10,2005-11-01,-0.059397,9.50,27.683,NaN,11,10.10,29.421,0.15000
...,...,...,...,...,...,...,...,...,...,...,...,...
421586,93436,2012-03-30,88160R10,2012-03-01,0.114640,37.24,3916.300,18.09700,11,33.41,3494.800,13.73900
421587,93436,2012-04-30,88160R10,2012-04-01,-0.110370,33.13,3485.700,13.02700,11,37.24,3916.300,18.09700
421588,93436,2012-05-31,88160R10,2012-05-01,-0.109570,29.50,3103.800,10.79800,11,33.13,3485.700,13.02700
421589,93436,2012-06-29,88160R10,2012-06-01,0.060678,31.29,3295.600,10.87400,11,29.50,3103.800,10.79800


In [201]:
# creating portfolios
bins = [-.00001, 1, 4, np.inf]
df["bins"] = df.groupby('caldt').feelag.transform(pd.cut, bins=bins, labels=False)

ew = (df.groupby(['caldt', 'bins']).ret.mean().unstack(level='bins')
      .rename('p{:.0f}'.format,axis='columns')*100)

ew['p3'] = ew.p0 - ew.p2
ew

bins,p0,p1,p2,p3
caldt,,,,
2003-01-31,-2.496359,-0.820190,-12.350338,9.853978
2003-02-28,-2.718891,-5.618787,-8.693200,5.974309
2003-03-31,1.719020,0.704056,-12.071568,13.790588
2003-04-30,10.841523,15.249342,15.888082,-5.046559
2003-05-30,11.946824,25.422119,27.656257,-15.709433
...,...,...,...,...
2012-03-30,3.481743,4.116197,0.810106,2.671638
2012-04-30,-1.264333,0.719475,-3.582510,2.318176
2012-05-31,-6.440576,-9.385430,-11.498005,5.057428


In [202]:
from finance_byu.summarize import summary
summary(ew)

bins,p0,p1,p2,p3
count,115.000000,115.000000,115.000000,115.000000
mean,1.230941,0.912352,-0.934087,2.165029
std,6.127127,7.660437,8.819907,4.527893
tstat,2.154415,1.277197,-1.135723,5.127627
pval,0.033312,0.204128,0.258455,0.000001
min,-22.157510,-21.053019,-22.563000,-15.709433
25%,-1.836873,-3.425434,-6.520955,-0.195664
50%,1.718496,0.719475,-0.728106,2.871914
75%,4.291088,4.950120,3.103033,5.106093
max,22.701813,25.422119,27.656257,13.790588


As shown above, the tstat for the portfolios shows that p0 and p3 are statistically significant portfolios with average returns much higher than 0.

## Question 2:

In [203]:
fac = pd.read_csv("14-factors.csv", parse_dates=['caldt'])

fac

,caldt,exmkt,smb,hml,umd,rf
0,1927-01-31,-0.06,-0.37,4.54,0.36,0.25
1,1927-02-28,4.18,0.04,2.94,-2.14,0.26
2,1927-03-31,0.13,-1.65,-2.61,3.61,0.30
3,1927-04-30,0.46,0.30,0.81,4.30,0.25
4,1927-05-31,5.44,1.53,4.73,3.00,0.30
...,...,...,...,...,...,...
1159,2023-08-31,-2.39,-3.16,-1.06,3.77,0.45
1160,2023-09-29,-5.24,-2.51,1.52,0.26,0.43
1161,2023-10-31,-3.19,-3.87,0.19,1.73,0.47
1162,2023-11-30,8.84,-0.02,1.64,2.75,0.44


In [204]:
ew = ew.join(fac.set_index('caldt'),how='inner')
ew

,p0,p1,p2,p3,exmkt,smb,hml,umd,rf
caldt,,,,,,,,,
2003-01-31,-2.496359,-0.820190,-12.350338,9.853978,-2.57,1.33,-0.83,1.55,0.10
2003-02-28,-2.718891,-5.618787,-8.693200,5.974309,-1.88,-0.45,-1.39,1.17,0.09
2003-03-31,1.719020,0.704056,-12.071568,13.790588,1.09,1.02,-1.93,1.48,0.10
2003-04-30,10.841523,15.249342,15.888082,-5.046559,8.22,0.63,1.14,-9.33,0.10
2003-05-30,11.946824,25.422119,27.656257,-15.709433,6.05,4.67,0.40,-10.70,0.09
...,...,...,...,...,...,...,...,...,...
2012-03-30,3.481743,4.116197,0.810106,2.671638,3.11,-0.64,1.14,1.30,0.00
2012-04-30,-1.264333,0.719475,-3.582510,2.318176,-0.85,-0.42,-0.78,3.75,0.00
2012-05-31,-6.440576,-9.385430,-11.498005,5.057428,-6.19,0.07,-1.07,6.49,0.01


In [205]:
names = ['exp0', 'exp1', 'exp2', 'exp3']
ew[names] = ew[[s[2:] for s in names]].sub(ew['rf'], axis='index')

ew

,p0,p1,p2,p3,exmkt,smb,hml,umd,rf,exp0,exp1,exp2,exp3
caldt,,,,,,,,,,,,,
2003-01-31,-2.496359,-0.820190,-12.350338,9.853978,-2.57,1.33,-0.83,1.55,0.10,-2.596359,-0.920190,-12.450337,9.753978
2003-02-28,-2.718891,-5.618787,-8.693200,5.974309,-1.88,-0.45,-1.39,1.17,0.09,-2.808891,-5.708787,-8.783200,5.884309
2003-03-31,1.719020,0.704056,-12.071568,13.790588,1.09,1.02,-1.93,1.48,0.10,1.619020,0.604056,-12.171568,13.690588
2003-04-30,10.841523,15.249342,15.888082,-5.046559,8.22,0.63,1.14,-9.33,0.10,10.741523,15.149342,15.788082,-5.146559
2003-05-30,11.946824,25.422119,27.656257,-15.709433,6.05,4.67,0.40,-10.70,0.09,11.856824,25.332119,27.566257,-15.799433
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2012-03-30,3.481743,4.116197,0.810106,2.671638,3.11,-0.64,1.14,1.30,0.00,3.481743,4.116197,0.810106,2.671638
2012-04-30,-1.264333,0.719475,-3.582510,2.318176,-0.85,-0.42,-0.78,3.75,0.00,-1.264333,0.719475,-3.582510,2.318176
2012-05-31,-6.440576,-9.385430,-11.498005,5.057428,-6.19,0.07,-1.07,6.49,0.01,-6.450576,-9.395430,-11.508005,5.047428


In [206]:
reg = [smf.ols(f'{y} ~ exmkt + smb + hml + umd',data=ew).fit() for y in names]

In [207]:
from finance_byu.regtables import Regtable

Regtable(reg,stat='tstat',sig='coeff').render()

,exp0,exp1,exp2,exp3
Intercept,0.249**,-0.168,-2.095***,2.200***
,(2.47),(-0.49),(-4.78),(5.48)
exmkt,1.027***,1.003***,1.039***,-0.012
,(36.96),(10.50),(8.61),(-0.11)
smb,0.774***,1.102***,1.255***,-0.473**
,(15.73),(6.51),(5.88),(-2.42)
hml,0.014,0.009,0.059,-0.052
,(0.31),(0.06),(0.31),(-0.29)
umd,-0.205***,-0.330***,-0.451***,0.242***
,(-9.34),(-4.38),(-4.74),(2.77)


### Four Factor Model Analysis:
Portfolio 0, 2, and 3 have non-zero statistically significant intercepts that imply the model cannot explain the returns of those given portfolios. Portfolio number 2 and 3 in particular have extremely large unexplained returns. 

### 3:
What do the factor loadings from the four factor model regression about portfolio 2 (high lagged loan fee portfolio) tell you about the characteristics (small-cap/large cap, value/growth, etc) of the stocks in the high loan fee portfolio. Explain.



Portfolio 2 has a large, non-zero alpha of -2.095 with an extremely significant t-stat of -4.78. The SMB factor loading implies that the high loan fee portfolio has more small market cap stocks compared to large cap stocks. The HML factor loading is close to zero and not statistically significant. This shows that the portfolio is relatively balanced between value and growth stocks. Finally the negative UMD factor loading shows that the portfolio is largely made of previously losing stocks.

4: 4. Form portfolios double sorted on lagged loan fee and average analyst dispersion from the previous two months. You should use independent sorts to create these double sort portfolios (as in Fama and French (1992)). The analyst dispersion breakpoints should be *terciles* (A set of data arranged in order with values that partition the data into three groups, each containing one-third of the total data.). For the loan fees use the three portfolio categories you created in question #1. Therefore, your program should create **9 portfolios**. Report summary statistics for your portfolios (including a t-test of whether the average return is statistically different from zero for each portfolio). Make your portfolio sample period be January 2003 to July 2012. Note, some of your portfolios may contain a few missing values. Don't worry about them. The regression estimation in the next questions should work just fine.

In [208]:
df.mdt = df.caldt.values.astype('datetime64[M]')

In [209]:
data = df.merge(ibes,on=['cusip','mdt'],how='left')

In [210]:
data['displag'] = (data.groupby('permno')['disp']
                   .rolling(2).mean()
                   .reset_index(level=0, drop=True)
                   .groupby(data['permno']).shift(1))

data = data.query("displag == displag").reset_index(drop=True)

In [211]:
data['dispBins'] = data.groupby('caldt').displag.transform(pd.qcut,3,labels=False)

ew1 = data.groupby(['caldt', 'bins', 'dispBins']).ret.mean()*100

In [212]:
ew1 = ew1.unstack(level=[1, 2])

In [213]:
summary(ew1)

bins               0                                   1              \
dispBins           0           1           2           0           1   
count     113.000000  113.000000  113.000000  113.000000  113.000000   
mean        1.107959    1.295638    1.299859    0.673994    0.767201   
std         5.128617    6.330233    7.912607    7.615986    8.365396   
tstat       2.296480    2.175721    1.746289    0.940739    0.974904   
pval        0.023508    0.031677    0.083503    0.348863    0.331709   
min       -20.733140  -22.297178  -26.997964  -19.932046  -20.359796   
25%        -0.935950   -1.754578   -2.463951   -4.107000   -5.029467   
50%         1.995510    1.895857    1.931023   -0.012736   -0.034719   
75%         3.818620    4.457075    5.293325    4.483680    5.417606   
max        17.340272   23.812039   31.282847   32.005392   24.848950   

bins                           2                          
dispBins           2           0           1           2  
count     113.000000  109.000000  112.000000  113.000000  
mean        0.846254   -1.056012   -0.750963   -1.054263  
std        10.416573   12.764893   10.495270   11.417477  
tstat       0.863605   -0.863704   -0.757240   -0.981562  
pval        0.389651    0.389665    0.450510    0.328431  
min       -25.230651  -60.694000  -34.552000  -28.771825  
25%        -5.295200   -6.318033   -6.880485   -6.519681  
50%         0.356877   -0.746607   -0.127723   -2.002432  
75%         5.461641    5.505611    4.921129    3.902775  
max        48.946802   35.060000   44.831882   33.258260

In [214]:
ew1 = ew1.stack(level=1)
ew1 = ew1.join(fac.set_index('caldt'),how='inner')

In [215]:
ew1disp0 = ew1.query("dispBins == 0")
ew1disp1 = ew1.query("dispBins == 1")
ew1disp2 = ew1.query("dispBins == 2")

names = ['exp0', 'exp1', 'exp2']
names1 = [f"Disp{i}{name}" for i in range(3) for name in names]

ew1disp0[names1[0:3]] = ew1disp0[[int(s[3:]) for s in names]].sub(ew['rf'], axis='index')
ew1disp1[names1[3:6]] = ew1disp1[[int(s[3:]) for s in names]].sub(ew['rf'], axis='index')
ew1disp2[names1[6:9]] = ew1disp2[[int(s[3:]) for s in names]].sub(ew['rf'], axis='index')

In [216]:
reg1 = [smf.ols(f'Disp0{y} ~ exmkt + smb + hml + umd',data=ew1disp0).fit() for y in names]
reg1 += [smf.ols(f'Disp1{y} ~ exmkt + smb + hml + umd',data=ew1disp1).fit() for y in names]
reg1 += [smf.ols(f'Disp2{y} ~ exmkt + smb + hml + umd',data=ew1disp2).fit() for y in names]

In [217]:
from finance_byu.regtables import Regtable

Regtable(reg1,stat='tstat',sig='coeff').render()

,Disp0exp0,Disp0exp1,Disp0exp2,Disp1exp0,Disp1exp1,Disp1exp2,Disp2exp0,Disp2exp1,Disp2exp2
Intercept,0.213**,-0.381,-2.280**,0.224**,-0.425,-2.052***,0.033,-0.492,-2.467***
,(2.53),(-1.07),(-2.25),(2.40),(-1.04),(-3.28),(0.23),(-0.92),(-3.71)
exmkt,0.929***,0.831***,1.104***,1.075***,1.040***,1.033***,1.240***,1.250***,1.157***
,(40.29),(8.55),(4.03),(42.10),(9.31),(6.06),(31.51),(8.51),(6.35)
smb,0.548***,1.118***,1.327***,0.798***,1.224***,1.390***,1.053***,1.112***,1.594***
,(13.43),(6.51),(2.70),(17.67),(6.20),(4.61),(15.13),(4.28),(4.94)
hml,-0.015,0.172,0.024,-0.013,0.184,-0.067,0.107,0.067,0.233
,(-0.41),(1.12),(0.06),(-0.32),(1.04),(-0.25),(1.72),(0.29),(0.81)
umd,-0.104***,-0.456***,-0.285,-0.189***,-0.291***,-0.648***,-0.310***,-0.635***,-0.556***
,(-5.75),(-5.98),(-1.33),(-9.41),(-3.32),(-4.79),(-10.04),(-5.50),(-3.88)


Five out of the nine portfolios create statistically significant unexplained returns with the four factor model. Every high shorting fee portfolio has a negative alpha < -2.052 with a tstat higher than 2.25. The four other portfolios created insignificant unexplained returns that were close to 0. The 4 factor model does well with portfolios that have low shorting fees. The unexplained returns in these portfolios are much closer to zero and are not nearly as significant as the high fee portfolios. The greatest unexplained returns can be found in the high dispersion high short selling fee portfolio which aligns with the idea that high dispersion indicates lower future returns and the idea that securities with higher short selling fees have lower expected returns. The findings reject the idea of the four factor model being a valid model for securities with higher market friction, but we run into joint hypothesis theory where either the model is incorrect or the market is simply inefficient.

5. Test whether the *four* factor model holds with respect to the double sort portfolios in question #3. You should summarize your regressions results in one table. Explain what you can infer about validity of the four factor model based on these regression results.

6. Do you think Questions #4 and #5 represent a good test of the Miller (1997) model? Explain why or why not? If yes, are the results in questions #4 and #5 consistent with the Miller model?

7. DMS's (2003) best test of the Miller model is when they create portfolios double sorted on size (lagged market-cap) and analyst dispersion. Give one reason why double sort portfolios formed on loan fees and analyst dispersion may represent a superior test than double sort portfolios formed on size and analyst dispersion, and one reason it may represent an inferior test.

## 6:

Yes, questions 4 and 5 represent good tests of the Miller 1977 model. Questions 4 and 5 involve looking at a portfolio double sorted on short selling fees and analyst dispersion. The Miller model argues that when there is a higher degree of disagreement between analysts and there are contingencies involved with short selling, securities can become overpriced and will have lower returns over time. Questions 4 and 5 look at these variables and sort the returns in a way where we can see how accurate the Miller model really is. Question 4 aligns well with the miller model - the returns of the portfolio with high disperion and high fees has a mean return of -1.05% while the returns on the portfolio with high disperion and low fees has a mean return of 1.3%. This perfectly agrees with the miller model since it relies on the assumption of market friction creating mispricings in the market. When we look at the regression results, they align with the miller model perfectly. The lowest alpha at -2.467% is the portfolio with the highest dispersion and the highest fees. The unexplained returns are also much closer to zero when short selling fees are smaller which like the data from question 4 is perfectly aligned with the miller model. This implies that when the short selling fees are lower, the market is able to be more efficient since the miller model models inefficiency and relies on the friction of short selling. 

## 7:

Why loan fees and analyst dispersion could be a superior test:
The Miller model relies on the assumption of short selling friction when making the argument for the mispricing to occur in the market. Using loan fees creates a direct look at the friction of short selling and allows us to see how the higher friction effects returns on securities with different levels of analyst dispersion. In theory, using the short selling fees should be the perfect way to test the Miller model because it is the exact variable that miller theorized his model from. 

Why loan fees and analyst dispersion may be an inferior test:
Double sorting the portfolios based on security size and analyst dispersion captures does a good job of capturing more of the frictions of short selling. Typically higher market cap securities are much easier and cheaper to short sell compared to small market cap securities. While loan fees are one part of short selling friction, the access to higher liquidity and other things like this make short selling a large-cap security much nicer than short selling a small-cap security. Furthermore, looking at the loading for the SMB risk factor, it had a very large effect in each portfolio that had been regressed. The SMB risk factor actually had the largest loading in over half of the portfolios. This implies that the SMB risk factor has a very big effect on the actual market returns in the portfolios sorted with loan fees and analyst dispersion. Sorting the securities on size and dispersion allow the real market returns to more accurately reflect the miller model because it takes care of the largest risk factor found in the loan fee double sort aswell as capturing other short selling contigencies not included in the short selling loan fee. 